In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [4]:
spark = SparkSession.builder \
.appName("PySpark Save Example") \
.master("local[*]") \
.getOrCreate()

26/08/14 15:46:51 WARN Utils: Your hostname, DESKTOP-IVIE3LC resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/14 15:46:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/14 15:46:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
df = spark.read.csv("../12/Subjects.csv", header=True, inferSchema=True)
df = df.withColumn("total", col("kor") + col("eng") + col("math") + col("science"))
df.show()

+-----+----+---+---+----+-------+-----+
|class|name|kor|eng|math|science|total|
+-----+----+---+---+----+-------+-----+
|    1| aaa| 67| 87|  90|     98|  342|
|    1| bbb| 45| 45|  56|     98|  244|
|    1| ccc| 95| 59|  96|     88|  338|
|    1| ddd| 65| 94|  89|     98|  346|
|    1| eee| 45| 65|  78|     98|  286|
|    1| fff| 78| 76|  98|     89|  341|
|    2| ggg| 87| 67|  65|     56|  275|
|    2| hhh| 89| 98|  78|     78|  343|
|    2| iii|100| 78|  56|     65|  299|
|    2| jjj| 99| 89|  87|     87|  362|
|    2| kkk| 98| 45|  56|     54|  253|
|    2| lll| 65| 89|  87|     78|  319|
+-----+----+---+---+----+-------+-----+



In [6]:
# 평균점수 계산
df.groupBy().avg("total").show()

# 조건 필터링 : 총점 300이상
df.filter(col("total") >= 300).show()

+-----------------+
|       avg(total)|
+-----------------+
|312.3333333333333|
+-----------------+

+-----+----+---+---+----+-------+-----+
|class|name|kor|eng|math|science|total|
+-----+----+---+---+----+-------+-----+
|    1| aaa| 67| 87|  90|     98|  342|
|    1| ccc| 95| 59|  96|     88|  338|
|    1| ddd| 65| 94|  89|     98|  346|
|    1| fff| 78| 76|  98|     89|  341|
|    2| hhh| 89| 98|  78|     78|  343|
|    2| jjj| 99| 89|  87|     87|  362|
|    2| lll| 65| 89|  87|     78|  319|
+-----+----+---+---+----+-------+-----+



In [7]:
# Temp View 생성
df.createOrReplaceTempView("students")

# SQL 쿼리
high_score = spark.sql("SELECT name, total FROM students WHERE total >= 300")
high_score.show()

+----+-----+
|name|total|
+----+-----+
| aaa|  342|
| ccc|  338|
| ddd|  346|
| fff|  341|
| hhh|  343|
| jjj|  362|
| lll|  319|
+----+-----+



In [8]:
spark.stop()
print("SparkSession 종료")

SparkSession 종료


# pyspark 데이터분석 + 머신러닝

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.classification import DecisionTreeClassifier

In [16]:
spark = SparkSession.builder \
.appName("PySpark_ML_Example") \
.master("local[*]") \
.getOrCreate()

print("SparkSession 생성 완료")

SparkSession 생성 완료


In [18]:
# header은 첫번째 row 를 컬럼명으로 사용
# inferSchema 는 데이터 타입 자동 추론

df = spark.read.csv("../12/Subjects.csv", header=True, inferSchema=True)

print("csv 파일 로드 완료")
df.show()

csv 파일 로드 완료
+-----+----+---+---+----+-------+
|class|name|kor|eng|math|science|
+-----+----+---+---+----+-------+
|    1| aaa| 67| 87|  90|     98|
|    1| bbb| 45| 45|  56|     98|
|    1| ccc| 95| 59|  96|     88|
|    1| ddd| 65| 94|  89|     98|
|    1| eee| 45| 65|  78|     98|
|    1| fff| 78| 76|  98|     89|
|    2| ggg| 87| 67|  65|     56|
|    2| hhh| 89| 98|  78|     78|
|    2| iii|100| 78|  56|     65|
|    2| jjj| 99| 89|  87|     87|
|    2| kkk| 98| 45|  56|     54|
|    2| lll| 65| 89|  87|     78|
+-----+----+---+---+----+-------+



In [20]:
# 데이터 전처리 및 파생 변수 생성
# 결측값은 0으로 대체
# 총점 (total) 및 합격여부(pass) 컬럼 생성

df = df.fillna({"kor": 0, "eng" : 0, "math" : 0, "science" : 0}) # 결측값은 0으로 대체
df = df.withColumn("total", col("kor") + col("eng") + col("math") + col("science"))
df = df.withColumn("pass", when(col("total") >= 300, 1).otherwise(0))

print("데이터 전처리 완료")
df.show()

데이터 전처리 완료
+-----+----+---+---+----+-------+-----+----+
|class|name|kor|eng|math|science|total|pass|
+-----+----+---+---+----+-------+-----+----+
|    1| aaa| 67| 87|  90|     98|  342|   1|
|    1| bbb| 45| 45|  56|     98|  244|   0|
|    1| ccc| 95| 59|  96|     88|  338|   1|
|    1| ddd| 65| 94|  89|     98|  346|   1|
|    1| eee| 45| 65|  78|     98|  286|   0|
|    1| fff| 78| 76|  98|     89|  341|   1|
|    2| ggg| 87| 67|  65|     56|  275|   0|
|    2| hhh| 89| 98|  78|     78|  343|   1|
|    2| iii|100| 78|  56|     65|  299|   0|
|    2| jjj| 99| 89|  87|     87|  362|   1|
|    2| kkk| 98| 45|  56|     54|  253|   0|
|    2| lll| 65| 89|  87|     78|  319|   1|
+-----+----+---+---+----+-------+-----+----+



In [24]:
# 회기 모델(linear Regression)
# - 입력피처(x) : kor, eng, math, science
# - 타겟(y) : total
# --> 새로운 입력값을 받고 학습 결과 기반으로 총점을 예측하는 모델

assembler = VectorAssembler(
  inputCols=["kor", "eng", "math", "science"],
  outputCol="features"
)
# Spark ML은 feature를 벡터 형태로 받아야 학습 가능
# 여러 컬럼(feature1, feature2.....)를 한 컬럼(feature) 에 합침
train_df = assembler.transform(df).select("features", "total") # x, y set

# 회기 모델 생성 및 학습
lr = LinearRegression(featuresCol="features", labelCol="total")
lr_model = lr.fit(train_df)

#예측 수행
# 학습된 모델로 df의 label예측

lr_predictions = lr_model.transform(train_df)
print("Linear Regression 모델 학습 완료")

lr_predictions.select("features", "total", "prediction").show(5)

26/08/14 16:28:24 WARN Instrumentation: [fdfeea31] regParam is zero, which might cause numerical instability and overfitting.


Linear Regression 모델 학습 완료
+--------------------+-----+------------------+
|            features|total|        prediction|
+--------------------+-----+------------------+
|[67.0,87.0,90.0,9...|  342| 342.0000000000004|
|[45.0,45.0,56.0,9...|  244|243.99999999999918|
|[95.0,59.0,96.0,8...|  338|338.00000000000153|
|[65.0,94.0,89.0,9...|  346| 346.0000000000002|
|[45.0,65.0,78.0,9...|  286|285.99999999999903|
+--------------------+-----+------------------+
only showing top 5 rows



In [25]:
# 분류 모델(Logistic Regression)
# - 입력 피처 동일
# - 타겟 : pass
# - 목적 : 합격(1)/불합격(0) 분류

assembler2 = VectorAssembler(
  inputCols=["kor", "eng", "math", "science"],
  outputCol="features"
)

train_df2 = assembler2.transform(df).select("features", "pass")

# 분류 모델 생성 및 학습
logr = LogisticRegression(featuresCol="features", labelCol="pass")
logr_model = logr.fit(train_df2)

# 에측 수행
logr_predictions = logr_model.transform(train_df2)
print("LogisticRegression 모델 학습 완료")

logr_predictions.select("features", "pass", "prediction", "probability").show(5, truncate=False)

LogisticRegression 모델 학습 완료
+---------------------+----+----------+------------------------------------------+
|features             |pass|prediction|probability                               |
+---------------------+----+----------+------------------------------------------+
|[67.0,87.0,90.0,98.0]|1   |1.0       |[2.81944823643108E-9,0.9999999971805518]  |
|[45.0,45.0,56.0,98.0]|0   |0.0       |[1.0,0.0]                                 |
|[95.0,59.0,96.0,88.0]|1   |1.0       |[1.245296963688586E-8,0.9999999875470303] |
|[65.0,94.0,89.0,98.0]|1   |1.0       |[1.852718822498173E-10,0.9999999998147281]|
|[45.0,65.0,78.0,98.0]|0   |0.0       |[0.9999999769882363,2.3011763716773714E-8]|
+---------------------+----+----------+------------------------------------------+
only showing top 5 rows

